<a href="https://colab.research.google.com/github/hanbiphyun/ESSA_YB/blob/main/ESAA_OB_week10_1_RNN_%EC%8B%A4%EC%8A%B5%ED%95%84%EC%82%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 문자단위 RNN 다대다 구조 구현
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [3]:
# 훈련 데이터 전처리하기
# 입력 데이터와 레이블 데이터에 대해 문자 집합 만들기
input_str= 'apple'
label_str= 'pple!'
char_vocab= sorted(list(set(input_str+label_str)))
vocab_size= len(char_vocab)
print('문자 집합의 크기 : {}'.format(vocab_size))

문자 집합의 크기 : 5


- 하이퍼파라미터 정의
- 원핫벡터를 사용할것이므로 입력 크기는 문자 집합 크기여야함

In [4]:
input_size= vocab_size # 입력크기는 문자 집합의 크기
hidden_size=5
output_size=5
learning_rate=0.1

- 문자집합에 고유한 정수 부여

In [5]:
char_to_index= dict((c,i) for i,c in enumerate(char_vocab))
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


- 반대로 정수로부터 문자 얻을 수 있는 index_to_char만들기

In [6]:
index_to_char= {}
for key, value in char_to_index.items():
    index_to_char[value]= key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


- 입력 데이터와 레이블 데이터의 각 문자들을 정수로 맵핑

In [10]:
x_data= [char_to_index[c] for c in input_str]
y_data= [char_to_index[c] for c in label_str]
print(x_data)  # apple
print(y_data)  # pple!

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


- 파이토치의 nn.RNN()은 기본적으로 3차원 텐서 입력받음 -> 배치 차원 추가
- 텐서연산인 unsqueeze(0)를 통해 해결할수도있음

In [14]:
# 3차원으로 만들어주는것
x_data= [x_data]
y_data= [y_data]
print(x_data)
print(y_data)

[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


In [15]:
# 입력 시퀀스의 각 문자들을 원핫 벡터로 바꿔줌
# 단위행렬 생성-> 행 추출
x_one_hot= [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


In [17]:
X= torch.FloatTensor(x_one_hot)
Y= torch.LongTensor(y_data)

In [18]:
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([1, 5, 5])
레이블의 크기 : torch.Size([1, 5])


####모델 구현하기

In [19]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        # RNN셀 구현
        self.rnn= torch.nn.RNN(input_size, hidden_size, batch_first=True)
        #출력층 구현
        self.fc= torch.nn.Linear(hidden_size, output_size, bias=True)

    # 구현한 RNN셀과 출력층 연결
    def forward(self, x):
        x, _status= self.rnn(x)
        x= self.fc(x)
        return x

In [20]:
net= Net(input_size, hidden_size, output_size)

In [21]:
outputs= net(X)
print(outputs.shape)
# 배치 차원, 시점, 출력크기

torch.Size([1, 5, 5])


In [22]:
# 2차원 텐서로 변환
print(outputs.view(-1, input_size).shape)

torch.Size([5, 5])


In [23]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 5])
torch.Size([5])


- 옵티마이저와 손실함수 정의

In [24]:
criterion= torch.nn.CrossEntropyLoss()
optimizer= optim.Adam(net.parameters(), learning_rate)

In [25]:
# 학습
for i in range(100):
    optimizer.zero_grad()
    outputs= net(X)
    # 배치 차원 제거를 위해 view해줌
    loss= criterion(outputs.view(-1, input_size), Y.view(-1))
    loss.backward()  #기울기계산
    optimizer.step()   # 아까 옵티마이저에 넣어둔 파라미터 업데이트

    # 최종 예측값인 각 time-step별 5차원 벡터에 대해 가장 높은 값의 인덱스 선택
    result= outputs.data.numpy().argmax(axis=2)
    result_str= ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(i, 'loss:', loss.item(), 'prediction: ', result, 'true Y:', y_data, 'prediction str:', result_str)


0 loss: 1.5967185497283936 prediction:  [[4 4 4 4 4]] true Y: [[4, 4, 3, 2, 0]] prediction str: ppppp
1 loss: 1.324991226196289 prediction:  [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
2 loss: 1.151599407196045 prediction:  [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
3 loss: 0.9840871095657349 prediction:  [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
4 loss: 0.835841178894043 prediction:  [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
5 loss: 0.6881107687950134 prediction:  [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
6 loss: 0.5487061738967896 prediction:  [[4 4 4 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pppe!
7 loss: 0.43799275159835815 prediction:  [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pple!
8 loss: 0.34927454590797424 prediction:  [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] prediction str: pple!
9 loss: 0.27333736419677734 prediction:  [[4 4 3 2 0]] true Y: [[4, 4, 3, 2, 0]] pr

##2. 더 많은 데이터로 학습한 문자단위 RNN

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim

1. 훈련 데이터 전처리하기
- 문자 집합을 생성하고 각 문자에 고유한 정수를 부여함

In [37]:
sentence = ("if you want to build a ship, don't drum up people together to collect wood and don't assign them tasks and work, but rather teach them to long for the endless immensity of the sea.")


In [38]:
# 중복을 제거한 문자 집합 생성
char_set= list(set(sentence))
# 각 문자에 정수 인코딩
char_dic= {c: i for i, c in enumerate(char_set)}

In [39]:
print(char_dic) # 공백도 하나의 원소

{'f': 0, 'h': 1, 'n': 2, ',': 3, 'a': 4, 'y': 5, 'b': 6, 't': 7, 's': 8, 'i': 9, "'": 10, 'r': 11, 'm': 12, 'l': 13, 'k': 14, 'u': 15, 'd': 16, 'g': 17, 'e': 18, 'c': 19, 'o': 20, 'w': 21, '.': 22, ' ': 23, 'p': 24}


In [40]:
dic_size= len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기 : 25


- 원핫벡터로 사용할 것이므로 입력크기가 25
- 하이퍼 파라미터 설정
- sequence_length :샘플을 10개 단위로 끊어서 샘플 만들것

In [41]:
hidden_size= dic_size
sequence_length=10
learning_rate=0.1

In [42]:
# 데이터 구성
x_data=[]
y_data=[]

for i in range(0, len(sentence)- sequence_length):
    x_str= sentence[i:i+sequence_length]
    # 한 단어씩 밀리는 것 보여줌
    y_str= sentence[i+1:i+sequence_length+1]
    print(i, x_str, '->', y_str)

    # x str to index
    x_data.append([char_dic[c] for c in x_str])
    y_data.append([char_dic[c] for c in y_str])

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

In [43]:
print(x_data[0])
print(y_data[0])

[9, 0, 23, 5, 20, 15, 23, 21, 4, 2]
[0, 23, 5, 20, 15, 23, 21, 4, 2, 7]


- 입력 시퀀스에 대해서 원-핫 인코딩 수행하고 입력 데이터와 레이블 데이터를 텐서 변환

In [44]:
x_one_hot= [np.eye(dic_size)[x] for x in x_data]
X= torch.FloatTensor(x_one_hot)

# 파이토치텐서를 Long(int64)타입으로 변환
Y= torch.LongTensor(y_data)

In [45]:
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([170, 10, 25])
레이블의 크기 : torch.Size([170, 10])


In [46]:
print(X[0])

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,

In [47]:
print(Y[0]) #?

tensor([ 0, 23,  5, 20, 15, 23, 21,  4,  2,  7])


###2. 모델구현하기

In [48]:
class Net(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, layers): # 현재 hidden_size는 dic_size와 같음.
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x


In [50]:
# 은닉층 2개 쌓음
net=Net(dic_size, hidden_size,2)

In [51]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

- 모델에 입력넣고 출력크기 확인

In [52]:
outputs= net(X)
print(outputs.shape)

torch.Size([170, 10, 25])


In [53]:
print(outputs.view(-1, dic_size).shape) # 2차원 텐서로 변환

torch.Size([1700, 25])


In [54]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([170, 10])
torch.Size([1700])


In [55]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X) # (170, 10, 25) 크기를 가진 텐서를 매 에포크마다 모델의 입력으로 사용
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    # results의 텐서 크기는 (170, 10)
    results = outputs.argmax(dim=2)
    predict_str = ""
    for j, result in enumerate(results):
        if j == 0: # 처음에는 예측 결과를 전부 가져오지만
            predict_str += ''.join([char_set[t] for t in result])
        else: # 그 다음에는 마지막 글자만 반복 추가
            predict_str += char_set[result[-1]]

    print(predict_str)


hyhhnyyyyhhhhyyhhyhyhynyyhyhyyyyyhyyhyyhyyyyhyyyhyyyhhyyhyyyyyyyyhyyhyyyyyyyyhhyyyyhyyhyyhyyhyhhhyyhyyyyyhhyyyyyyhhhhyyhyhuyhhuhyyhyhhhyyyyyyhyyyyyhyhuhyyhhyyyyhhhyyyhhhyyyhyhuyyy
                                                                                                                                                                                   
t ynn ayn  yyn bya  a  yn  yt y t y    t y bb   b         b  t bn  y  yyn  yn  yy  yyt yt  a    by         yy   yn   yy  b       t     ynt b   yn  ya  y   yn  yy   ya     t    ya 
                                                                                                                                                                                   
o to s to o o e  o    s  o  o o  o  o o      o o ee    o     oo o s  t  s  o      o o o e             o o       o o   o o  o  s  o s  so o  o e         o s  oe   o o    o s  o o  
roteesetseooo so se seto se seseetd seotoeeessmsesoetomoeseoeto semseso setoeoeeemoedeseeseeeesetots